In [ ]:
# Switch path to root of project
import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"

import torch
from urllib.request import urlopen

from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer

# Load the model and config files from the Hugging Face Hub
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')


In [2]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = model.to(device)
model = model.eval()

In [ ]:
# load abdomen vqa data
import json
# vqa_train_file = "/data/xxx/hallucination/CARES/IU_Xray/training.json"
# vqa_train_file = "/data/xxx/hallucination/CARES/OmniMedVQA/training_masks_top4.json"
# vqa_train_file = "/data/xxx/hallucination/PathVQA/pvqa/training_masks_top4.json"
# vqa_train_file = "/data/xxx/hallucination/VQA_RAD/data/training_masks_top4.json"
vqa_train_file = "/data/xxx/hallucination/Slake/data/training_masks_all.json"

# vqa_train_file = "/data/xxx/hallucination/IU_Xray/data_report/training.json"
# vqa_train_file = "/data/xxx/hallucination/MIMIC_CXR/data_report/training.json"


with open(vqa_train_file, "r") as f:
    vqa_data = json.load(f)
# Check the loaded data
print(f"Loaded {len(vqa_data)} vqa data")
# filter only english data
vqa_data_train = vqa_data
print(f"Filtered {len(vqa_data_train)} vqa data")

Loaded 4919 vqa data
Filtered 4919 vqa data


In [ ]:
# sample images
import os
import random
from pathlib import Path

from sklearn.manifold import TSNE
import numpy as np

import matplotlib.pyplot as plt

# root_dir = Path("/data/xxx/hallucination/IU_Xray/iu_xray/images")
# root_dir = Path("/data/xxx/hallucination/OmniMedVQA/VQA/raw/OmniMedVQA")
# root_dir = Path("/data/xxx/hallucination/PathVQA/pvqa/images/train")
root_dir = Path("/data/xxx/hallucination/Slake/imgs")
# root_dir = Path("/data/xxx/hallucination/VQA_RAD/images/")
# image_paths = list(root_dir.rglob("source.jpg"))

# root_dir = Path("/data/xxx/hallucination/IU_Xray/iu_xray/images")
# root_dir = Path("/data/xxx/hallucination/MIMIC_CXR/sampled_files_train")

image_paths = list(set([i['image'] for i in vqa_data_train]))

# Step 2: Randomly sample 100 images
sampled_paths = image_paths

print(f"Found {len(image_paths)} training images, sampled {len(sampled_paths)}")

Found 450 training images, sampled 450


In [5]:
full_sampled_paths = []
for path in sampled_paths:
    full_path = os.path.join(root_dir, path)
    full_sampled_paths.append(full_path)

In [6]:
import torch.nn.functional as F
from tqdm import tqdm

In [7]:
def interpolate_attention_map(attn_map: torch.Tensor, src_H: int = 14, tgt_H: int = 24) -> torch.Tensor:
    assert attn_map.shape[0] == src_H * src_H

    # Step 1: reshape to [1, 1, H, W]
    attn_2d = attn_map.view(1, 1, src_H, src_H)

    # Step 2: interpolate to target resolution
    attn_interp = F.interpolate(attn_2d, size=(tgt_H, tgt_H), mode='bilinear', align_corners=False)

    # Step 3: flatten to [tgt_H * tgt_H]
    return attn_interp.view(tgt_H * tgt_H)

In [ ]:
feature_cos_matrix = []
start_index, sample_number = 0, len(sampled_paths)
# start_index, sample_number = 0, 10
ind = 0
cos_matrix_dict = {}
cos_matrix_before_dict = {}
# sample = "/data/xxx/hallucination/Slake/imgs/xmlab44/source.jpg"
for img_path in tqdm(full_sampled_paths[start_index:start_index+sample_number]):  # label: organ/lesion/other
    relative_path = sampled_paths[ind]
    ind += 1
    inputs = preprocess(Image.open(img_path)).to("cuda").unsqueeze(0)  # [1, 3, 224, 224]
    with torch.no_grad():
        image_features, attentions = model.encode_image(inputs)
    patch_tokens = image_features.squeeze(0)[1:, :]  # [1, 196, D]
    patch_tokens_norm = F.normalize(patch_tokens, dim=1)  # Normalize the features
    cos_sim_before = patch_tokens_norm @ patch_tokens_norm.T
    # STEP 1: Reshape from [196, D] to [14, 14, D]
    patch_H, patch_W = 14, 14  # original grid
    D = patch_tokens.shape[1]
    patch_tokens_2d = patch_tokens.view(patch_H, patch_W, D).permute(2, 0, 1).unsqueeze(0)  # [1, D, 14, 14]

    # STEP 2: Interpolate to [24, 24]
    upsampled = F.interpolate(patch_tokens_2d, size=(24, 24), mode='bilinear', align_corners=False)  # [1, D, 24, 24]

    # STEP 3: Flatten to [576, D]
    patch_tokens_up = upsampled.squeeze(0).permute(1, 2, 0).reshape(-1, D)  # [576, D]
    patch_tokens_up = F.normalize(patch_tokens_up, dim=1)  # Normalize again

    # STEP 4: Compute cosine similarity
    cos_sim_up = patch_tokens_up @ patch_tokens_up.T  # [576, 576]

    # Store result
    feature_cos_matrix.append(cos_sim_up.cpu())
    cos_matrix_dict[relative_path] = cos_sim_up.cpu().numpy()
    cos_matrix_before_dict[relative_path] = cos_sim_before.cpu().numpy()


100%|██████████| 450/450 [00:13<00:00, 34.43it/s]


In [14]:
cos_matrix_dict.keys()

dict_keys(['xmlab11/source.jpg', 'xmlab376/source.jpg', 'xmlab290/source.jpg', 'xmlab123/source.jpg', 'xmlab384/source.jpg', 'xmlab36/source.jpg', 'xmlab202/source.jpg', 'xmlab519/source.jpg', 'xmlab541/source.jpg', 'xmlab292/source.jpg'])

In [ ]:
# save cosine similarity matrix
import pickle
# save_root_path = "/data/xxx/hallucination/CARES/IU_Xray/steering_data_biomed/"
# save_root_path = "/data/xxx/hallucination/CARES/OmniMedVQA/steering_data_biomed/"
save_root_path = "/data/xxx/hallucination/PathVQA/pvqa/steering_data_biomed/"
# save_root_path = "/data/xxx/hallucination/Slake/steering/steer_data_biomed/"
# save_root_path = "/data/xxx/hallucination/VQA_RAD/steering/steer_data_biomed/"
# save_root_path = "/data/xxx/hallucination/IU_Xray/data_report/steering_data_biomed/"
# save_root_path = "/data/xxx/hallucination/MIMIC_CXR/data_report/steering_data_biomed/"


if not os.path.exists(save_root_path):
    os.makedirs(save_root_path)
# save cosine similarity matrix
with open(save_root_path + "cosine_matrix_dict.pkl", "wb") as f:
    pickle.dump(cos_matrix_dict, f)